# Stage 3 Feature Backbone Sweep Clean

Train-CV-only feature sweep for crop-level coarse classification. The notebook evaluates several frozen visual backbones with the same LogisticRegression protocol, selects hyperparameters only on train folds, evaluates once on clean val, and saves a compact archive.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, csv, traceback
from collections import Counter

import pandas as pd
import numpy as np
from IPython.display import display

RUN_NAME = "stage3_feature_backbone_sweep_clean"
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_feature_backbone_sweep_clean.tar.gz")
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
MODELS = [
    {"name":"clip_b32", "model_id":"openai/clip-vit-base-patch32", "batch_size":32},
    {"name":"clip_l14", "model_id":"openai/clip-vit-large-patch14", "batch_size":16},
    {"name":"siglip_b16_224", "model_id":"google/siglip-base-patch16-224", "batch_size":16},
    {"name":"dinov2_base", "model_id":"facebook/dinov2-base", "batch_size":16},
]
C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
print("RUN_NAME:", RUN_NAME)


In [ ]:
def sh(args, check=True):
    print("$", " ".join(str(a) for a in args))
    p=subprocess.run([str(a) for a in args], text=True, capture_output=True)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path):
    rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def resolve_crop(row, jsonl_path):
    p=Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    base=Path(jsonl_path).parent
    for c in [base/p, base.parent/p, DATA_ROOT/p]:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this sweep")
sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate", "scikit-learn", "tabulate"], check=True)


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root/TRAIN_JSONL_REL).exists() and (root/VAL_JSONL_REL).exists():
        DATA_ROOT=root; break
if DATA_ROOT is None:
    raise FileNotFoundError("stage3_regrouped_v2 train/val JSONL was not found")
train_jsonl=DATA_ROOT/TRAIN_JSONL_REL
val_jsonl=DATA_ROOT/VAL_JSONL_REL
train_rows=[r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
val_rows=[r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
print("DATA_ROOT:", DATA_ROOT)
print("train:", len(train_rows), Counter(r["coarse_class"] for r in train_rows))
print("val:", len(val_rows), Counter(r["coarse_class"] for r in val_rows))
train_paths=[resolve_crop(r, train_jsonl) for r in train_rows]
val_paths=[resolve_crop(r, val_jsonl) for r in val_rows]
y_train=np.array([r["coarse_class"] for r in train_rows])
y_val=np.array([r["coarse_class"] for r in val_rows])
val_ids=[r["record_id"] for r in val_rows]


In [ ]:
import torch
from PIL import Image
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
from transformers import AutoModel, AutoProcessor, AutoImageProcessor

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)

def load_processor(model_id):
    try:
        return AutoProcessor.from_pretrained(model_id)
    except Exception:
        return AutoImageProcessor.from_pretrained(model_id)

def image_features(model, inputs):
    if hasattr(model, "get_image_features"):
        return model.get_image_features(**inputs)
    out = model(**inputs)
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    return out.last_hidden_state.mean(dim=1)

def embed(model_id, paths, batch_size):
    device="cuda" if torch.cuda.is_available() else "cpu"
    processor=load_processor(model_id)
    model=AutoModel.from_pretrained(model_id).to(device)
    model.eval()
    feats=[]
    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch_paths=paths[start:start+batch_size]
            images=[Image.open(p).convert("RGB") for p in batch_paths]
            inputs=processor(images=images, return_tensors="pt", padding=True).to(device)
            out=image_features(model, inputs).float()
            out=torch.nn.functional.normalize(out, dim=-1)
            feats.append(out.cpu().numpy())
            print(model_id, "embedded", min(start+batch_size, len(paths)), "/", len(paths))
    del model
    torch.cuda.empty_cache()
    return np.concatenate(feats, axis=0)

def metrics(y_true, y_pred):
    recalls=recall_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "recall_insulator_ok": float(recalls[0]),
        "recall_defect_flashover": float(recalls[1]),
        "recall_defect_broken": float(recalls[2]),
    }

def cv_select(X, y):
    splitter=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows=[]
    for C in C_GRID:
        for cw in [None, "balanced"]:
            fold=[]
            for tr, va in splitter.split(X, y):
                clf=make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight=cw, max_iter=5000, random_state=42))
                clf.fit(X[tr], y[tr])
                pred=clf.predict(X[va])
                fold.append(metrics(y[va], pred))
            row={
                "C": C,
                "class_weight": cw or "none",
                "cv_accuracy_mean": float(np.mean([m["accuracy"] for m in fold])),
                "cv_macro_f1_mean": float(np.mean([m["macro_f1"] for m in fold])),
                "cv_macro_f1_std": float(np.std([m["macro_f1"] for m in fold])),
                "cv_flashover_recall_mean": float(np.mean([m["recall_defect_flashover"] for m in fold])),
                "cv_broken_recall_mean": float(np.mean([m["recall_defect_broken"] for m in fold])),
            }
            rows.append(row)
    rows=sorted(rows, key=lambda r:(r["cv_macro_f1_mean"], r["cv_flashover_recall_mean"], r["cv_broken_recall_mean"]), reverse=True)
    return rows[0], rows


In [ ]:
all_rows=[]
for spec in MODELS:
    name=spec["name"]; model_id=spec["model_id"]; batch_size=spec["batch_size"]
    out_dir=WORK_DIR/name
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        print("===", name, model_id, "===")
        X_train=embed(model_id, train_paths, batch_size)
        X_val=embed(model_id, val_paths, batch_size)
        selected, cv_rows = cv_select(X_train, y_train)
        pd.DataFrame(cv_rows).to_csv(out_dir/"cv_results.csv", index=False)
        cw=None if selected["class_weight"]=="none" else selected["class_weight"]
        clf=make_pipeline(StandardScaler(), LogisticRegression(C=float(selected["C"]), class_weight=cw, max_iter=5000, random_state=42))
        clf.fit(X_train, y_train)
        pred=clf.predict(X_val)
        probs=clf.predict_proba(X_val)
        val_metrics=metrics(y_val, pred)
        cm=confusion_matrix(y_val, pred, labels=LABELS)
        pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(out_dir/"confusion_matrix.csv")
        pred_rows=[]
        for rid, gt, pr, prob in zip(val_ids, y_val, pred, probs):
            row={"record_id":rid,"gt":gt,"pred_coarse_class":pr,"confidence":float(np.max(prob)),"correct":gt==pr}
            for cls,val in zip(clf.classes_, prob): row[f"prob_{cls}"]=float(val)
            pred_rows.append(row)
        pd.DataFrame(pred_rows).to_csv(out_dir/"val_predictions.csv", index=False)
        result={"name":name,"model_id":model_id,"status":"ok",**selected,**val_metrics}
        write_json(out_dir/"result_summary.json", result)
        print(result)
        all_rows.append(result)
    except Exception as exc:
        err={"name":name,"model_id":model_id,"status":"error","error":str(exc)}
        (out_dir/"error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print(traceback.format_exc())
        all_rows.append(err)

summary=pd.DataFrame(all_rows)
summary.to_csv(WORK_DIR/"backbone_sweep_summary.csv", index=False)
display(summary)


In [ ]:
ok = pd.DataFrame(all_rows)
ok2 = ok[ok["status"]=="ok"].copy()
if len(ok2):
    ok2 = ok2.sort_values(["macro_f1", "accuracy", "recall_defect_flashover", "recall_defect_broken"], ascending=False)
    best = ok2.iloc[0].to_dict()
else:
    best = {}
report_lines=[
    "# Stage 3 Feature Backbone Sweep Clean",
    "",
    "Selection protocol: 5-fold stratified train CV by macro-F1. Validation is evaluated once per backbone.",
    "",
    "## Summary",
    ok.to_markdown(index=False),
    "",
    "## Best candidate",
    json.dumps(best, indent=2),
]
(WORK_DIR/"report.md").write_text("\n".join(report_lines), encoding="utf-8")
print((WORK_DIR/"report.md").read_text(encoding="utf-8"))

if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(WORK_DIR.parent), WORK_DIR.name], check=True)
print("Archive:", ARCHIVE_PATH)
